Master Drug Repository

In [1]:
import requests
import pandas as pd
import time
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Configuration

In [2]:
OPENFDA_BASE = "https://api.fda.gov/drug/ndc.json"

PAGE_SIZE = 100

OUTPUT_FOLDER = "repository"

Create Repository Folder

In [3]:
if not os.path.exists(
    OUTPUT_FOLDER
):
    os.makedirs(
        OUTPUT_FOLDER
    )

print(
    f"Repository folder: "
    f"{os.path.abspath(OUTPUT_FOLDER)}"
)

Repository folder: C:\Users\barry.peterson\OneDrive - Integra Connect LLC\Documents\Pharmacy Reports\Master Drug File\repository


OpenFDA Downloaded

In [4]:
def download_openfda_ndc_records(
    max_records=10000
):

    records = []

    skip = 0

    while len(records) < max_records:

        print(
            f"Downloading records "
            f"{skip} - "
            f"{skip + PAGE_SIZE}"
        )

        try:

            response = requests.get(

                OPENFDA_BASE,

                params={

                    "limit":
                        PAGE_SIZE,

                    "skip":
                        skip
                },

                timeout=60
            )

            response.raise_for_status()

            batch = (
                response.json()
                .get(
                    "results",
                    []
                )
            )

            if len(batch) == 0:

                break

            records.extend(
                batch
            )

            skip += PAGE_SIZE

            time.sleep(0.25)

        except Exception as e:

            print(
                f"Download Error: {e}"
            )

            break

    return records[:max_records]


Test Downloader

In [5]:
import time

start_time = time.time()

records = download_openfda_ndc_records(
    max_records=10000
)

elapsed_time = (
    time.time() - start_time
)

print(
    f"Records Downloaded: "
    f"{len(records):,}"
)

print(
    f"Elapsed Seconds: "
    f"{elapsed_time:.1f}"
)

print(
    f"Elapsed Minutes: "
    f"{elapsed_time / 60:.2f}"
)

Records Downloaded: 10,000
Elapsed Seconds: 71.7
Elapsed Minutes: 1.20


Product Builder

In [6]:
def build_products_df(
    records
):

    rows = []

    for record in records:

        rows.append({

            "product_ndc":
                record.get(
                    "product_ndc"
                ),

            "brand_name":
                record.get(
                    "brand_name"
                ),

            "generic_name":
                record.get(
                    "generic_name"
                ),

            "manufacturer":
                record.get(
                    "labeler_name"
                ),

            "dosage_form":
                record.get(
                    "dosage_form"
                ),

            "route":
                ", ".join(
                    record.get(
                        "route",
                        []
                    )
                ),

            "marketing_category":
                record.get(
                    "marketing_category"
                ),

            "application_number":
                record.get(
                    "application_number"
                ),

            "product_type":
                record.get(
                    "product_type"
                )
        })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Package Builder

In [7]:
def build_packages_df(
    records
):

    rows = []

    for record in records:

        product_ndc = (
            record.get(
                "product_ndc"
            )
        )

        for package in record.get(
            "packaging",
            []
        ):

            rows.append({

                "product_ndc":
                    product_ndc,

                "package_ndc":
                    package.get(
                        "package_ndc"
                    ),

                "package_description":
                    package.get(
                        "description"
                    ),

                "sample":
                    package.get(
                        "sample"
                    )
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Ingredient Builder

In [8]:
def build_ingredients_df(
    records
):

    rows = []

    for record in records:

        product_ndc = (
            record.get(
                "product_ndc"
            )
        )

        for ingredient in record.get(
            "active_ingredients",
            []
        ):

            rows.append({

                "product_ndc":
                    product_ndc,

                "ingredient_name":
                    ingredient.get(
                        "name"
                    ),

                "strength":
                    ingredient.get(
                        "strength"
                    )
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Build Repository Tables

In [9]:
products_df = build_products_df(
    records
)

packages_df = build_packages_df(
    records
)

ingredients_df = build_ingredients_df(
    records
)

print(
    f"Products: {len(products_df):,}"
)

print(
    f"Packages: {len(packages_df):,}"
)

print(
    f"Ingredients: {len(ingredients_df):,}"
)

Products: 9,932
Packages: 18,592
Ingredients: 15,774


Save Repository Assets

In [10]:
products_df.to_parquet(
    f"{OUTPUT_FOLDER}/products.parquet",
    index=False
)

packages_df.to_parquet(
    f"{OUTPUT_FOLDER}/packages.parquet",
    index=False
)

ingredients_df.to_parquet(
    f"{OUTPUT_FOLDER}/ingredients.parquet",
    index=False
)

print("Repository assets saved")

Repository assets saved


Repository Profile

In [11]:
print(
    "Unique Products:",
    products_df[
        "product_ndc"
    ].nunique()
)

print(
    "Unique Packages:",
    packages_df[
        "package_ndc"
    ].nunique()
)

print(
    "Unique Ingredients:",
    ingredients_df[
        "ingredient_name"
    ].nunique()
)

print(
    "Unique Manufacturers:",
    products_df[
        "manufacturer"
    ].nunique()
)

Unique Products: 9915
Unique Packages: 18592
Unique Ingredients: 2741
Unique Manufacturers: 1646


Build Repository Table

In [12]:
def build_repository_df(
    products_df,
    packages_df,
    ingredients_df
):

    repository_df = products_df.merge(

        packages_df,

        on="product_ndc",

        how="left"
    )

    repository_df = repository_df.merge(

        ingredients_df,

        on="product_ndc",

        how="left"
    )

    return (
        repository_df
        .drop_duplicates()
        .reset_index(drop=True)
    )

Create Repository

In [13]:
repository_df = build_repository_df(

    products_df,

    packages_df,

    ingredients_df
)

print(
    f"Repository Rows: "
    f"{len(repository_df):,}"
)

display(
    repository_df.head(25)
)

Repository Rows: 26,102


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,application_number,product_type,package_ndc,package_description,sample,ingredient_name,strength
0,59116-3393,None,Methylphenidate Hydrochloride,"Cambrex Charles City, Inc",POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59116-3393-5,60 kg in 1 DRUM (59116-3393-5),None,METHYLPHENIDATE HYDROCHLORIDE,1 kg/kg
1,59651-022,None,Loperamide Hydrochloride,Aurobindo Pharma Limited,TABLET,,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,59651-022-49,4000 TABLET in 1 BAG (59651-022-49),None,LOPERAMIDE HYDROCHLORIDE,2 mg/1
2,59651-860,None,Ubrogepant,Aurobindo Pharma Limited,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59651-860-37,50 kg in 1 DRUM (59651-860-37),None,UBROGEPANT,50 kg/50kg
3,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,HYDROCHLOROTHIAZIDE,12.5 mg/1
4,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,LOSARTAN POTASSIUM,100 mg/1
5,60870-0288,None,Naloxone Hydrochloride,Aspen Oss B.V.,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,60870-0288-0,1 BAG in 1 DRUM (60870-0288-0) / 21000 g in 1 BAG,None,NALOXONE HYDROCHLORIDE,1 g/g
6,61662-0015,None,Degarelix Acetate,"Lianyungang Runzhong Pharmaceutical Co., Ltd.",POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,61662-0015-1,4 kg in 1 BAG (61662-0015-1),None,DEGARELIX ACETATE,1 kg/kg
7,62207-020,None,RIFAXIMIN,Granules India Limited,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,62207-020-02,25 kg in 1 DRUM (62207-020-02),None,RIFAXIMIN,1 kg/kg
8,62512-0024,None,CARBAMAZEPINE,Amoli Organics (A Division of Umedica Laboratories Pvt.Ltd.),POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,62512-0024-1,50 kg in 1 DRUM (62512-0024-1),None,CARBAMAZEPINE,1 kg/kg
9,62938-0004,None,Loratadine,CATALENT U.K. SWINDON ZYDIS LIMITED,"TABLET, ORALLY DISINTEGRATING",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,62938-0004-1,"3024 BLISTER PACK in 1 BOX (62938-0004-1) / 10 TABLET, ORALLY DISINTEGRATING in 1 BLISTER PACK",None,LORATADINE,5 mg/1


Repository Profile

In [14]:
repository_profile = {

    "rows":
        len(repository_df),

    "products":
        repository_df[
            "product_ndc"
        ].nunique(),

    "packages":
        repository_df[
            "package_ndc"
        ].nunique(),

    "ingredients":
        repository_df[
            "ingredient_name"
        ].nunique(),

    "manufacturers":
        repository_df[
            "manufacturer"
        ].nunique()
}

for key, value in repository_profile.items():

    print(
        f"{key}: {value:,}"
    )

rows: 26,102
products: 9,915
packages: 18,592
ingredients: 2,741
manufacturers: 1,646


Save Repository Asset

In [15]:
repository_df.to_parquet(

    f"{OUTPUT_FOLDER}/repository.parquet",

    index=False
)

print(
    "repository.parquet saved"
)

repository.parquet saved


Repository Metadata

In [16]:
from datetime import datetime
import json

In [17]:
metadata = {

    "build_timestamp":

        datetime.now()
        .strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "product_count":

        int(
            products_df[
                "product_ndc"
            ].nunique()
        ),

    "package_count":

        int(
            packages_df[
                "package_ndc"
            ].nunique()
        ),

    "ingredient_count":

        int(
            ingredients_df[
                "ingredient_name"
            ].nunique()
        ),

    "manufacturer_count":

        int(
            products_df[
                "manufacturer"
            ].nunique()
        )
}

Save Metadata

In [18]:
metadata_file = (

    f"{OUTPUT_FOLDER}/"
    "repository_metadata.json"
)

with open(
    metadata_file,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print(
    f"Saved: {metadata_file}"
)

print(
    json.dumps(
        metadata,
        indent=4
    )
)

Saved: repository/repository_metadata.json
{
    "build_timestamp": "2026-06-30 19:26:43",
    "product_count": 9915,
    "package_count": 18592,
    "ingredient_count": 2741,
    "manufacturer_count": 1646
}


Repository Loader

In [19]:
def load_repository():

    return pd.read_parquet(

        f"{OUTPUT_FOLDER}/"
        "repository.parquet"
    )

Test Repository Load

In [20]:
repo = load_repository()

print(
    f"Rows: {len(repo):,}"
)

display(
    repo.head()
)

Rows: 26,102


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,application_number,product_type,package_ndc,package_description,sample,ingredient_name,strength
0,59116-3393,None,Methylphenidate Hydrochloride,"Cambrex Charles City, Inc",POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59116-3393-5,60 kg in 1 DRUM (59116-3393-5),None,METHYLPHENIDATE HYDROCHLORIDE,1 kg/kg
1,59651-022,None,Loperamide Hydrochloride,Aurobindo Pharma Limited,TABLET,,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,59651-022-49,4000 TABLET in 1 BAG (59651-022-49),None,LOPERAMIDE HYDROCHLORIDE,2 mg/1
2,59651-860,None,Ubrogepant,Aurobindo Pharma Limited,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59651-860-37,50 kg in 1 DRUM (59651-860-37),None,UBROGEPANT,50 kg/50kg
3,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,HYDROCHLOROTHIAZIDE,12.5 mg/1
4,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,LOSARTAN POTASSIUM,100 mg/1


Enhanced Build Statistics

In [21]:
import os

def get_file_size_mb(filepath):

    if not os.path.exists(filepath):
        return 0

    return round(
        os.path.getsize(filepath) /
        (1024 * 1024),
        2
    )

Repository Asset Report

In [22]:
asset_report = pd.DataFrame({

    "asset": [

        "products.parquet",
        "packages.parquet",
        "ingredients.parquet",
        "repository.parquet"
    ],

    "size_mb": [

        get_file_size_mb(
            f"{OUTPUT_FOLDER}/products.parquet"
        ),

        get_file_size_mb(
            f"{OUTPUT_FOLDER}/packages.parquet"
        ),

        get_file_size_mb(
            f"{OUTPUT_FOLDER}/ingredients.parquet"
        ),

        get_file_size_mb(
            f"{OUTPUT_FOLDER}/repository.parquet"
        )
    ]
})

asset_report

,asset,size_mb
0,products.parquet,0.38
1,packages.parquet,0.58
2,ingredients.parquet,0.18
3,repository.parquet,1.09


Repository Health Report

In [23]:
health_report = {

    "products":
        products_df[
            "product_ndc"
        ].nunique(),

    "packages":
        packages_df[
            "package_ndc"
        ].nunique(),

    "ingredients":
        ingredients_df[
            "ingredient_name"
        ].nunique(),

    "manufacturers":
        products_df[
            "manufacturer"
        ].nunique(),

    "repository_rows":
        len(repository_df)
}

for k, v in health_report.items():

    print(
        f"{k}: {v:,}"
    )

products: 9,915
packages: 18,592
ingredients: 2,741
manufacturers: 1,646
repository_rows: 26,102


Estimate Full Repository Size

In [24]:
sample_records = len(records)

repository_rows = len(
    repository_df
)

packages = packages_df[
    "package_ndc"
].nunique()

print(
    f"Sample records: "
    f"{sample_records:,}"
)

print(
    f"Repository rows: "
    f"{repository_rows:,}"
)

print(
    f"Packages per product: "
    f"{packages / sample_records:.2f}"
)


Sample records: 10,000
Repository rows: 26,102
Packages per product: 1.86


Build Log

In [25]:
from datetime import datetime

build_log = {

    "build_time":
        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "sample_records":
        len(records),

    "repository_rows":
        len(repository_df),

    "products":
        products_df[
            "product_ndc"
        ].nunique(),

    "packages":
        packages_df[
            "package_ndc"
        ].nunique(),

    "ingredients":
        ingredients_df[
            "ingredient_name"
        ].nunique()
}

build_log

{'build_time': '2026-06-30 19:26:43',
 'sample_records': 10000,
 'repository_rows': 26102,
 'products': 9915,
 'packages': 18592,
 'ingredients': 2741}

Save Build Log

In [26]:
with open(

    f"{OUTPUT_FOLDER}/build_log.json",

    "w"
) as f:

    json.dump(
        build_log,
        f,
        indent=4
    )

print(
    "build_log.json saved"
)

build_log.json saved


Scale Test

In [27]:
import time

start_time = time.time()

records_10000 = (
    download_openfda_ndc_records(
        max_records=10000
    )
)

elapsed = (
    time.time() - start_time
)

print(
    f"Downloaded "
    f"{len(records_10000):,} "
    f"records"
)

print(
    f"Elapsed Seconds: "
    f"{elapsed:.1f}"
)

Downloaded 10,000 records
Elapsed Seconds: 86.6


In [28]:
print("\n")
print("=" * 50)
print("REPOSITORY STATISTICS")
print("=" * 50)

print(
    f"Products: "
    f"{products_df['product_ndc'].nunique():,}"
)

print(
    f"Packages: "
    f"{packages_df['package_ndc'].nunique():,}"
)

print(
    f"Ingredients: "
    f"{ingredients_df['ingredient_name'].nunique():,}"
)

print(
    f"Manufacturers: "
    f"{products_df['manufacturer'].nunique():,}"
)

print(
    f"Repository Rows: "
    f"{len(repository_df):,}"
)



REPOSITORY STATISTICS
Products: 9,915
Packages: 18,592
Ingredients: 2,741
Manufacturers: 1,646
Repository Rows: 26,102


In [29]:
import os

for file_name in [

    "products.parquet",

    "packages.parquet",

    "ingredients.parquet",

    "repository.parquet"
]:

    full_file = (
        f"{OUTPUT_FOLDER}/"
        f"{file_name}"
    )

    size_mb = round(
        os.path.getsize(
            full_file
        ) / 1024 / 1024,
        2
    )

    print(
        f"{file_name}: "
        f"{size_mb} MB"
    )

products.parquet: 0.38 MB
packages.parquet: 0.58 MB
ingredients.parquet: 0.18 MB
repository.parquet: 1.09 MB


In [30]:
CONFIG = {

    "max_records": 10000,

    "page_size": 100,

    "output_folder": "repository"
}

In [31]:
repository_statistics = {

    "products":
        int(
            products_df[
                "product_ndc"
            ].nunique()
        ),

    "packages":
        int(
            packages_df[
                "package_ndc"
            ].nunique()
        ),

    "ingredients":
        int(
            ingredients_df[
                "ingredient_name"
            ].nunique()
        ),

    "manufacturers":
        int(
            products_df[
                "manufacturer"
            ].nunique()
        ),

    "repository_rows":
        int(
            len(repository_df)
        )
}

In [32]:
with open(

    f"{OUTPUT_FOLDER}/repository_statistics.json",

    "w"
) as f:

    json.dump(
        repository_statistics,
        f,
        indent=4
    )

print(
    "repository_statistics.json saved"
)

repository_statistics.json saved


Parquet

In [33]:
data_dictionary = pd.DataFrame({

    "column_name": repository_df.columns,

    "data_type":
        repository_df.dtypes.astype(str)
})

data_dictionary.to_csv(

    f"{OUTPUT_FOLDER}/repository_data_dictionary.csv",

    index=False
)

print(
    "repository_data_dictionary.csv saved"
)

repository_data_dictionary.csv saved


In [34]:
CONFIG = {

    "max_records": 10000,

    "page_size": 100,

    "output_folder": "repository"
}

In [35]:
repository_statistics = {

    "products":
        int(
            products_df[
                "product_ndc"
            ].nunique()
        ),

    "packages":
        int(
            packages_df[
                "package_ndc"
            ].nunique()
        ),

    "ingredients":
        int(
            ingredients_df[
                "ingredient_name"
            ].nunique()
        ),

    "manufacturers":
        int(
            products_df[
                "manufacturer"
            ].nunique()
        ),

    "repository_rows":
        int(
            len(repository_df)
        )
}

In [36]:
with open(

    f"{OUTPUT_FOLDER}/repository_statistics.json",

    "w"
) as f:

    json.dump(
        repository_statistics,
        f,
        indent=4
    )

print(
    "repository_statistics.json saved"
)

repository_statistics.json saved


In [37]:
data_dictionary = pd.DataFrame({

    "column_name": repository_df.columns,

    "data_type":
        repository_df.dtypes.astype(str)
})

In [38]:
data_dictionary.to_csv(

    f"{OUTPUT_FOLDER}/repository_data_dictionary.csv",

    index=False
)

print(
    "repository_data_dictionary.csv saved"
)

repository_data_dictionary.csv saved
